# Método: expectativa pitagórica en la ACB 25/26

Reproduce las cifras del artículo llamando al paquete `pitagorica`. Reglas en `docs/preregistro.md`.

In [1]:
import numpy as np, pandas as pd
from pitagorica import datos, pitagoras, simulacion, estadistica as est, K_NBA, SEMILLA

rng = np.random.default_rng(SEMILLA)
df = datos.cargar()
t, t1, t2 = datos.tabla_equipos(df), datos.tabla_equipos(df, 1), datos.tabla_equipos(df, 2)
t.head()

,G,V,PF,PC,pos,pct_v,net_rtg
team,,,,,,,
BAXI Manresa,34,16,2932,3080,2768.72,0.470588,-5.345430
Baskonia,34,25,3177,2950,2698.82,0.735294,8.411083
Bàsquet Girona,34,14,2913,3024,2672.46,0.411765,-4.153477
Casademont Zaragoza,34,10,2962,3176,2705.60,0.294118,-7.909521
Coviran Granada,34,6,2848,3153,2646.22,0.176471,-11.525875


## Calibrar k

In [2]:
k = pitagoras.ajustar_k_tabla(t)
print(
    "k ACB =",
    round(k, 2),
    "| error medio:",
    round(pitagoras.error_medio(t, k), 2),
    "| con k=14:",
    round(pitagoras.error_medio(t, K_NBA), 2),
)
print(pitagoras.validacion_mitades(t1, t2))
print(pitagoras.validacion_loo(t))

k ACB = 11.27 | error medio: 1.51 | con k=14: 1.55
{'k_vuelta1': 11.515056454555157, 'k_vuelta2': 12.132198432139344, 'error_acb': 1.0293439723913496, 'error_k14': 1.1272470732835762}
{'error_acb': 1.5997281288093965, 'error_k14': 1.5507042920976446, 'k_min': 10.841869729958708, 'k_max': 11.79402199612047}


## Desvíos y banda del azar (simulación log5)

In [3]:
pyth = pd.Series(pitagoras.pitagorico(t["PF"], t["PC"], k), index=t.index)
wins_sim, diag = simulacion.simular(datos.partidos(df), pyth, 2000, rng)
tabla = pd.DataFrame({"V": t["V"], "esperadas": t["G"] * pyth}).join(simulacion.banda(wins_sim))
tabla["desvio"] = tabla["V"] - tabla["esperadas"]
tabla.sort_values("desvio", ascending=False).round(2)

,V,esperadas,p_bajo,p_alto,desvio
team,,,,,
BAXI Manresa,16,12.40,9.0,17.0,3.60
Surne Bilbao Basket,19,15.68,11.0,20.0,3.32
Hiopos Lleida,12,9.74,6.0,15.0,2.26
Real Madrid,26,24.48,20.0,28.0,1.52
Baskonia,25,23.71,19.0,27.0,1.29
UCAM Murcia,25,24.12,19.0,28.0,0.88
Bàsquet Girona,14,13.47,9.0,18.0,0.53
Joventut,22,21.61,17.0,25.0,0.39
Río Breogán,15,14.66,11.0,19.0,0.34


## ¿Se repite? (2000 simulaciones aquí; el artículo usa 10.000: `make analisis`)

In [4]:
d1, d2 = est.desvio(t1, k), est.desvio(t2, k)
a = est.contraste_mitades(d1, d2, 2000, rng)
b = est.contraste_varianza(tabla["desvio"], tabla["esperadas"], wins_sim)
print(a)
print(b)
print(est.decision(a, b))

{'r': -0.015564258535496922, 'ic90_bajo': -0.3566749153396224, 'ic90_alto': 0.35874649161140926}
{'rms_observado': 1.8049587666653708, 'rms_azar_mediana': 2.556960887096542, 'rms_azar_p95': 3.332799773866702, 'p_valor': 0.9585}
{'cond_a': False, 'cond_b': False, 'conclusion': 'compatible_con_azar'}
